In [212]:
import jax_sn as jsn
%load_ext autoreload
%autoreload 2
import jax_sbgeom

import jax.numpy as jnp
import numpy as onp
from typing import Tuple
import jax
import numpy as onp
from functools import partial
jax.config.update("jax_enable_x64", True)

import pyvista as pv

def remove_zeros(array):
    a = onp.array(array)
    a[onp.abs(a) < 1e-14] = 0.0
    return a

from jax_sn.domain.geometric_domain import GeometricDomain, TopologicalSortedAngle
from jax_sn.solution_domain.solution_domain import FEMMatrices, PrecomputedGeometricData
from jax_sn.domain.geometric_domain import CrossSectionData, CrossSectionDataSingleGroup
from jax_sn.source import Source
from jax_sn.operators.transport_operator import InverseTransportOperatorBatched
from typing import List
from jax_sn.tools.mesh_tools import _pad_last_dimension
import matplotlib.pyplot as plt
from dataclasses import dataclass
onp.set_printoptions(linewidth = 200)
from finite_elements.test_moment_scatter_operator import _setup_all_sb

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [204]:
from jax_sn.domain.domain_generation import create_cartesian_setup
from jax_sn.solution_domain import BasixLagrangianSimplex
from jax_sn.solution_domain import SolutionDomain
from jax_sn.domain import CrossSectionData, Domain

In [247]:
dimension = 3
degree = 2 
tn_order = 5
solution_domain, quad_set, toposorts, sweep_array, source = _setup_all_sb(dimension, degree, tn_order)

In [264]:
transport_operator = InverseTransportOperatorBatched.from_solution_domain_data_and_toposort(solution_domain.solution_domain_data, toposorts, batch_size = 4)

moment_operator = jsn.operators.moment_operator.MomentOperator.from_quadrature_set(quad_set, 4)

In [267]:
@eqx.filter_jit
def apply_MTinv(moment_operator, transport_operator, source, energy_group):
    return moment_operator(transport_operator(source, energy_group))




In [269]:
%timeit apply_MTinv(moment_operator, transport_operator, source, 0)

99.9 ms ± 282 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [270]:
import equinox as eqx

@eqx.filter_jit
def call_transport_operator(transport_operator : InverseTransportOperatorBatched, source : Source, energy_group : 0 ):
    return transport_operator(source, energy_group)




In [259]:
@jax.jit
def call_transport_operator(transport_operator : InverseTransportOperatorBatched, source : Source, energy_group : 0 ):
    return transport_operator(source, energy_group)

In [260]:
%timeit call_transport_operator(transport_operator, source, 0)

ValueError: Non-hashable static arguments are not supported. An error occurred while trying to hash an object of type <class 'jax._src.interpreters.partial_eval.DynamicJaxprTracer'>, JitTracer<~int64[]>. The error was:
TypeError: unhashable type: 'DynamicJaxprTracer'


In [251]:
from jax_sn.operators.transport_operator import transport_inverse_sweep_multiple_angles_batched

In [256]:

def call_to2(solution_domain, toposorts, source, energy_group):
    buffer_sweep = jnp.zeros((toposorts.angle.shape[0], solution_domain.domain.n_elements, solution_domain.n_basis))
    return transport_inverse_sweep_multiple_angles_batched(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(energy_group), toposorts, source, buffer_sweep, 4)


%timeit call_to2(solution_domain, toposorts, source, 0)

97.2 ms ± 578 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [255]:
%timeit call_to2(solution_domain, toposorts, source, 0)
%timeit transport_inverse_sweep_multiple_angles_batched(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), toposorts, source, jnp.zeros((toposorts.angle.shape[0], solution_domain.domain.n_elements, solution_domain.n_basis)), 4)

99 ms ± 269 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
97.4 ms ± 276 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
#buffer_sweep = jnp.zeros((toposorts.angle.shape[0], solution_domain.domain.n_elements, solution_domain.n_basis))





97.4 ms ± 408 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)
